# Step 10: Register the Model

**SageMaker Unified Studio Component**: Model Registry

**What you'll learn**: Register your model for versioning and governance

In [1]:
import sagemaker
import os
from sagemaker.sklearn import SKLearnModel
from dotenv import load_dotenv

load_dotenv()
bucket_name = os.getenv('BUCKET_NAME')
role = os.getenv('EXECUTION_ROLE')

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


## Register Model

Model registration creates a versioned entry in the SageMaker Model Registry for governance and lineage tracking. The inference script is NOT needed at this stage - it's only required when deploying the model to an endpoint.

In [2]:
model_data = f's3://{bucket_name}/models/logistic_regression/model.tar.gz'

sklearn_model = SKLearnModel(
    model_data=model_data,
    role=role,
    framework_version='1.2-1',
    py_version='py3'
)

model_package = sklearn_model.register(
    content_types=['application/json'],
    response_types=['application/json'],
    inference_instances=['ml.t2.medium'],
    model_package_group_name='machine-overheat-models',
    approval_status='PendingManualApproval'
)

print(f"✓ Model registered: {model_package.model_package_arn}")

sagemaker.config INFO - Applied value from config key = SageMaker.Model.ExecutionRoleArn
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
✓ Model registered: arn:aws:sagemaker:eu-west-1:283784618425:model-package/machine-overheat-models/1


## Approve Model

Programmatically approve the registered model for deployment.

In [3]:
import boto3

sagemaker_client = boto3.client('sagemaker')

sagemaker_client.update_model_package(
    ModelPackageArn=model_package.model_package_arn,
    ModelApprovalStatus='Approved'
)

print(f'✓ Model approved: {model_package.model_package_arn}')

✓ Model approved: arn:aws:sagemaker:eu-west-1:283784618425:model-package/machine-overheat-models/1
